# Results table

Collects a set of experiment folders and builds a `pandas` table summarizing each run.

Each run folder is expected to contain:
- `.hydra/config.yaml` — the resolved Hydra configuration of the run;
- `run_data.json` — the metrics produced at the end of training (written by `main.py`).

The table reports, for every run:

| column | meaning |
|---|---|
| `target_type` | architecture of the target network (`Siren` or `MLP`) |
| `hyper_type` | architecture of the hypernetwork (always `MLP`) |
| `N` | number of training samples |
| `n_realizations` | number of variable samples sharing the same hypervariable |
| `hyper_layers` | number of linear layers of the hypernetwork |
| `target_layers` | number of linear layers of the (full) target network |
| `RRMSE_test` | relative RMSE on the test set |

In [ ]:
from pathlib import Path
import json
import yaml
import pandas as pd

## Configuration

Set `RESULTS_ROOT` to a directory that *contains* the run folders: it is searched
recursively for every `run_data.json` (Hydra nests runs as `runs_<problem>/<date>/<time>/`).

Alternatively, list explicit run folders in `RUN_FOLDERS` (this takes precedence).

In [ ]:
# Directory searched recursively for runs (each run = folder with .hydra/ and run_data.json).
RESULTS_ROOT = Path("../results")

# Optional: explicit list of run folders. If non-empty, RESULTS_ROOT is ignored.
RUN_FOLDERS = []

# Name of the test metric to report (as stored in run_data.json["test_metrics"]).
METRIC = "RRMSE"

## Helpers

In [ ]:
def find_run_folders(root, explicit):
    """Return the run folders to parse: the explicit list if given, otherwise every
    folder under `root` that holds both run_data.json and .hydra/config.yaml."""
    if explicit:
        return [Path(p) for p in explicit]
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"RESULTS_ROOT does not exist: {root.resolve()}")
    folders = {
        p.parent
        for p in root.rglob("run_data.json")
        if (p.parent / ".hydra" / "config.yaml").exists()
    }
    return sorted(folders)


def _load_yaml(path):
    with open(path) as f:
        return yaml.safe_load(f)


def _class_name(target):
    """'models.Siren' -> 'Siren'."""
    return str(target).split(".")[-1] if target else None


def _blocks_by_role(model_cfg, role_suffix):
    """All block dicts in the model config whose _target_ ends with role_suffix
    (e.g. 'TargetNetwork', 'Hypernetwork')."""
    return [
        v for v in model_cfg.values()
        if isinstance(v, dict) and str(v.get("_target_", "")).endswith(role_suffix)
    ]


def _num_layers(network):
    """A network whose num_neurons list has length L has L-1 linear layers.
    Only the length of the list is needed, so unresolved ${...} entries are fine."""
    n = (network or {}).get("num_neurons")
    return (len(n) - 1) if isinstance(n, list) else None

In [ ]:
def parse_run(folder):
    """Read one run folder and return a dict of summary fields."""
    folder = Path(folder)
    cfg = _load_yaml(folder / ".hydra" / "config.yaml")
    with open(folder / "run_data.json") as f:
        run = json.load(f)

    model = cfg.get("model", {}) or {}
    target_blocks = _blocks_by_role(model, "TargetNetwork")
    hyper_blocks = _blocks_by_role(model, "Hypernetwork")

    # Target network type (Siren vs MLP); if split across blocks they share the type.
    target_types = {
        _class_name((b.get("network") or {}).get("_target_")) for b in target_blocks
    }
    target_types.discard(None)
    target_type = "/".join(sorted(target_types)) or None

    # Hypernetwork type (always MLP, but derived for safety).
    hyper_types = {
        _class_name((b.get("network") or {}).get("_target_")) for b in hyper_blocks
    }
    hyper_types.discard(None)
    hyper_type = "/".join(sorted(hyper_types)) or "MLP"

    # Layer counts. The full target network may be split over several blocks
    # (e.g. target_first_layer + target_other_layers), so sum their layers.
    per_target = [_num_layers(b.get("network")) for b in target_blocks]
    per_target = [x for x in per_target if x is not None]
    target_layers = sum(per_target) if per_target else None
    hyper_layers = _num_layers(hyper_blocks[0].get("network")) if hyper_blocks else None

    # Data settings (present for the toy problem; absent for turbulence).
    train = (cfg.get("data_source", {}) or {}).get("train", {}) or {}
    N = train.get("N")
    n_realizations = train.get("n_realizations")

    # Test metric.
    rrmse = (run.get("test_metrics", {}) or {}).get(METRIC)

    return {
        "run": folder.name,
        "target_type": target_type,
        "hyper_type": hyper_type,
        "N": N,
        "n_realizations": n_realizations,
        "hyper_layers": hyper_layers,
        "target_layers": target_layers,
        f"{METRIC}_test": rrmse,
        "path": str(folder),
    }

## Build the table

In [ ]:
folders = find_run_folders(RESULTS_ROOT, RUN_FOLDERS)
print(f"Found {len(folders)} run folder(s).")

rows = []
for folder in folders:
    try:
        rows.append(parse_run(folder))
    except Exception as e:
        print(f"[skip] {folder}: {type(e).__name__}: {e}")

columns = [
    "run", "target_type", "hyper_type", "N", "n_realizations",
    "hyper_layers", "target_layers", f"{METRIC}_test", "path",
]
df = pd.DataFrame(rows, columns=columns)
df

## Sorted view and export

Sort by test error and (optionally) save to CSV.

In [ ]:
df_sorted = df.sort_values(f"{METRIC}_test", na_position="last").reset_index(drop=True)
df_sorted

In [ ]:
# df_sorted.to_csv("results_table.csv", index=False)